# TP2 - Market Basket Analysis (Partie 2)

INF8111 - Fouille de données, Automne 2026

### Membres de l'équipe
    - Lyna Kettani, 2124257
    - Membre 2
    - Ndèye Maty Ndiaye, 2405114

In [ ]:
ROOT = "gs://tp2-inf8111-maty-2026/"

In [ ]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.sql("select 'spark' as hello ")
df.show()

## 4. MBA pour le dataset complet (10 points)

Comme vous l'avez probablement remarqué, même pour un ensemble de données moins volumineux, l'algorithme MBA est coûteux en calcul. Pour cette raison, cette fois-ci, nous allons répéter le processus, en utilisant Google Cloud Platform (GCP) pour créer un grand cluster. Toutes les instructions pour créer un cluster avec Spark et soumettre un travail seront expliquées lors du laboratoire. Dans tous les cas, vous devez lire les instructions données dans le document ``Instruction_GCP.pdf``.

Cette fois, nous travaillerons avec le fichier ``OBSERVATIONS.csv``, qui contient plusieurs millions d'observations.

En utilisant les lignes de ``OBSERVATIONS.csv``, créez un bloc de données où chaque ligne contient la liste des symptômes observés pour chaque patient, de manière similaire au GroceryDataset. Ensuite, exécutez l'algorithme MBA pour cet ensemble de transactions.
Le format devra permettre d'appeler l'ensemble des méthodes précédemment implémentées. (Vous pouvez réutiliser votre code de la section 3).

- Vous devez signaler le temps passé pour effectuer cette tâche.
- La sortie doit contenir le nom des symptômes.
- **Important** : la première colonne (patterns) doit être classée par ordre alphabétique.
- Afin d'accélérer l'exécution, vous pouvez garder ce second notebook uniquement dédié à cette partie et l'insérer dans le rendu final.

**Remarque importante : joignez des captures d'écran de votre sortie et de votre configuration de cluster.**

**ATTENTION : Dès que le cluster sera configuré, GCP commencera à utiliser les 50 $ de crédits disponibles. Veuillez donc supprimer le cluster ainsi que le bucket de stockage associé une fois l’exécution terminée, afin d’éviter toute consommation inutile de crédits. Le simple fait d’arrêter le cluster ne suffit pas à empêcher la consommation de crédits. Les ressources créées doivent être supprimées.**


### Configuration du cluster Dataproc

Le traitement a été réalisé sur Google Cloud Dataproc avec le cluster `cluster-e63f`. Les captures suivantes présentent les paramètres sélectionnés lors de sa création ainsi que sa configuration réelle.

| Paramètre | Valeur |
|---|---|
| Nom du cluster | `cluster-e63f` |
| Fournisseur cloud | Google Cloud Platform (GCP) |
| Région | `us-east1` |
| Zone | `us-east1-c` |
| Type de cluster | Standard |
| Type de machine | `n4-highmem-16` |
| Processeurs par machine | 16 vCPU |
| Mémoire par machine | 128 Go |
| Type de disque | `hyperdisk-balanced` |
| Disque par machine | 150 Go |
| Nœud pilote | 1 |
| Nœuds de calcul principaux | 2 |
| Nœuds de calcul secondaires | 0 |
| Autoscaling | Activé (minimum 2, maximum 10 nœuds de calcul) |
| Version de l'image | `2.3.37-debian12` |
| Lightning Engine | Activé |
| Composants facultatifs | Jupyter, Iceberg, Delta |
| Bucket Cloud Storage | `tp2-inf8111-maty-2026` |


![Type de machine et autoscaling](./images/image_cluster2.png)

![Configuration des nœuds](./images/image_cluster3.png)

![Composants et paramètres supplémentaires](./images/image_cluster4.png)

![Configuration supplémentaire du cluster](./images/image_cluster5.png)

![Nom du cluster et région](./images/image_cluster1.png)

**Configuration réelle du cluster Dataproc**

![Détails du cluster cluster-e63f](./images/image_cluster6.png)

In [ ]:
%%time

from pyspark.sql import functions as F

# Lire le fichier complet des observations
observations_df = spark.read.csv(
    ROOT + "OBSERVATIONS.csv",
    header=True,
    inferSchema=True
)

# Regrouper les symptômes par patient
transactions_df = (
    observations_df
    .groupBy("VAERS_ID")
    .agg(
        F.sort_array(
            F.collect_set("SYMPTOM_ID")
        ).alias("symptoms")
    )
)

# Afficher 5 transactions pour vérifier le résultat
transactions_df.show(5, truncate=False)

![Regroupement des symptômes par patient](./images/Capture%20d’écran%202026-10-29%20à%2020.31.15.png)

In [ ]:
%%time

# Convertir les transactions en RDD pour réutiliser les fonctions MBA
transactions_rdd = (
    transactions_df
    .select("symptoms")
    .rdd
    .map(lambda row: row["symptoms"])
)

print("RDD des transactions créé avec succès.")

![Résultats](./images/Capture%20d’écran%202026-10-29%20à%2020.36.11.png)

In [ ]:
%%time

import time
from itertools import combinations
from pyspark.sql import functions as F
from pyspark.sql import types as T

debut = time.perf_counter()

# ============================================================
# 1. CHARGER LES DONNÉES
# ============================================================

observations_mba = (
    spark.read.csv(
        ROOT + "OBSERVATIONS.csv",
        header=True,
        inferSchema=False
    )
    .select("VAERS_ID", "SYMPTOM_ID")
    .dropna()
)

symptoms_mba = (
    spark.read.csv(
        ROOT + "SYMPTOMS.csv",
        header=True,
        inferSchema=False
    )
    .select("SYMPTOM_ID", "SYMPTOM_NAME")
    .dropna()
    .dropDuplicates(["SYMPTOM_ID"])
)

# Dictionnaire ID -> nom du symptôme
noms_dict = {
    row["SYMPTOM_ID"]: row["SYMPTOM_NAME"]
    for row in symptoms_mba.collect()
}

noms_bc = spark.sparkContext.broadcast(noms_dict)

# ============================================================
# 2. CRÉER LES TRANSACTIONS : SYMPTÔMES PAR PATIENT
# ============================================================

transactions_mba = (
    observations_mba
    .groupBy("VAERS_ID")
    .agg(
        F.sort_array(
            F.collect_set("SYMPTOM_ID")
        ).alias("symptoms")
    )
)

transactions_mba_rdd = (
    transactions_mba
    .select("symptoms")
    .rdd
    .map(lambda row: row["symptoms"])
)

# ============================================================
# 3. FONCTIONS MBA DE LA SECTION 3
# ============================================================

def map_to_patterns(symptoms):
    products = sorted(set(symptoms))

    for taille in range(1, min(4, len(products) + 1)):
        for pattern in combinations(products, taille):
            yield (pattern, 1)


def reduce_patterns(v1, v2):
    return v1 + v2


def map_to_subpatterns(pattern):
    products, occurrences = pattern

    yield (products, (None, occurrences))

    if len(products) > 1:
        for i, product in enumerate(products):
            subpattern = products[:i] + products[i + 1:]

            yield (
                subpattern,
                (product, occurrences)
            )


def map_to_assoc_rules(rule):
    pattern, (support, associations) = rule

    if support is None or support == 0:
        return (pattern, [])

    return (
        pattern,
        [
            (product, count / support)
            for product, count in associations
        ]
    )

# ============================================================
# 4. CALCULER LES MOTIFS SUR TOUTES LES TRANSACTIONS
# ============================================================

patterns_mba_rdd = transactions_mba_rdd.flatMap(
    map_to_patterns
)

combined_patterns_mba_rdd = patterns_mba_rdd.reduceByKey(
    reduce_patterns
)

# ============================================================
# 5. CONSTRUIRE LES RÈGLES D'ASSOCIATION
#    SANS UTILISER groupByKey()
# ============================================================

subpatterns_mba_rdd = combined_patterns_mba_rdd.flatMap(
    map_to_subpatterns
)

def creer_combinateur(valeur):
    product, count = valeur

    if product is None:
        return (count, [])

    return (None, [(product, count)])


def ajouter_valeur(acc, valeur):
    support, associations = acc
    product, count = valeur

    if product is None:
        support = count
    else:
        associations.append((product, count))

    return (support, associations)


def fusionner_combinateurs(a, b):
    support_a, associations_a = a
    support_b, associations_b = b

    support = (
        support_a
        if support_a is not None
        else support_b
    )

    associations_a.extend(associations_b)

    return (support, associations_a)


combined_rules_mba_rdd = subpatterns_mba_rdd.combineByKey(
    creer_combinateur,
    ajouter_valeur,
    fusionner_combinateurs
)

assoc_rules_mba_rdd = combined_rules_mba_rdd.map(
    map_to_assoc_rules
)

# ============================================================
# 6. CONVERTIR LES IDS EN NOMS DE SYMPTÔMES
# ============================================================

def convertir_regle(regle):
    pattern, associations = regle

    dictionnaire = noms_bc.value

    noms_pattern = sorted(
        dictionnaire.get(str(x), str(x))
        for x in pattern
    )

    pattern_texte = ", ".join(noms_pattern)

    associations_noms = sorted(
        [
            (
                dictionnaire.get(str(symptome), str(symptome)),
                float(confiance)
            )
            for symptome, confiance in associations
        ],
        key=lambda x: x[0].casefold()
    )

    associations_texte = "; ".join(
        f"{nom} : {confiance:.6f}"
        for nom, confiance in associations_noms
    )

    return (
        pattern_texte,
        associations_texte
    )


regles_noms_rdd = assoc_rules_mba_rdd.map(
    convertir_regle
)

# ============================================================
# 7. EXÉCUTER LE MBA ET RÉCUPÉRER LES 20 PREMIERS PATTERNS
#    PAR ORDRE ALPHABÉTIQUE
# ============================================================

apercu = regles_noms_rdd.takeOrdered(
    20,
    key=lambda x: x[0].casefold()
)

temps_execution = time.perf_counter() - debut

# ============================================================
# 8. CRÉER LE TABLEAU DEMANDÉ
# ============================================================

schema_resultat = T.StructType([
    T.StructField("Patterns", T.StringType(), False),
    T.StructField("Associations", T.StringType(), False)
])

tableau_mba = spark.createDataFrame(
    apercu,
    schema=schema_resultat
)

# ============================================================
# 9. AFFICHER LES RÉSULTATS
# ============================================================

print("=" * 65)
print("MBA COMPLET — OBSERVATIONS.csv")
print("=" * 65)

print("Données utilisées : OBSERVATIONS.csv")
print("Motifs calculés : 1, 2 et 3 symptômes")
print(f"Temps d'exécution du MBA : {temps_execution:.2f} secondes")

print("\n20 premiers patterns par ordre alphabétique :")

tableau_mba.orderBy("Patterns").show(
    20,
    truncate=False
)

In [ ]:
print("=" * 65)
print("MBA COMPLET — OBSERVATIONS.csv")
print("=" * 65)

print(f"Temps d'exécution : {temps_execution:.2f} secondes")

print("\n20 premiers patterns par ordre alphabétique :")

tableau_mba.orderBy("Patterns").show(20, truncate=False)

![Résultats MBA — Capture 1](./images/Capture%20d’écran%202026-10-29%20à%2019.06.37.png)

![Résultats MBA — Capture 2](./images/Capture%20d’écran%202026-10-29%20à%2019.06.46.png)

### Résultats pour un symptôme

Après avoir exécuté le MBA pour la plus grande collection d'observations, sélectionnez et affichez les règles d'association (nom du symptôme et valeur d'association) du symptôme **d'ID 11320**, c'est-à-dire lorsque le symptôme est seul. La sortie doit être formatée dans un tableau, où chaque ligne contient les informations d'un symptôme associé.

Ce qui est attendu :
- Titre du tableau : ID du symptôme 11320 et son nom
- Nombre de lignes du tableau
- Tableau contenant 2 colonnes : Symptôme (Nom du symptôme associé) et Information (Niveau de confiance)
- **Important** : La première colonne (symptôme) doit être classée par ordre alphabétique

Signalez le temps d'exécution.

In [ ]:
%%time

import time
from pyspark.sql import functions as F

debut = time.perf_counter()

# ============================================================
# 1. Définir le symptôme demandé
# ============================================================

symptome_id = "11320"

# ============================================================
# 2. Lire les fichiers
# ============================================================

observations = (
    spark.read.csv(
        ROOT + "OBSERVATIONS.csv",
        header=True,
        inferSchema=False
    )
    .select("VAERS_ID", "SYMPTOM_ID")
    .dropna()
    .dropDuplicates(["VAERS_ID", "SYMPTOM_ID"])
)

symptoms = (
    spark.read.csv(
        ROOT + "SYMPTOMS.csv",
        header=True,
        inferSchema=False
    )
    .select("SYMPTOM_ID", "SYMPTOM_NAME")
    .dropna()
    .dropDuplicates(["SYMPTOM_ID"])
)

# ============================================================
# 3. Trouver le nom du symptôme
# ============================================================

ligne_symptome = (
    symptoms
    .filter(F.col("SYMPTOM_ID") == symptome_id)
    .select("SYMPTOM_NAME")
    .first()
)

nom_symptome = (
    ligne_symptome["SYMPTOM_NAME"]
    if ligne_symptome is not None
    else "Nom inconnu"
)

# ============================================================
# 4. Trouver les patients présentant le symptôme demandé
# ============================================================

patients_symptome = (
    observations
    .filter(F.col("SYMPTOM_ID") == symptome_id)
    .select("VAERS_ID")
    .distinct()
)

# ============================================================
# 5. Calculer le support du symptôme
# ============================================================

support_symptome = patients_symptome.count()

# ============================================================
# 6. Trouver les symptômes associés
# ============================================================

associations = (
    observations
    .join(
        patients_symptome,
        on="VAERS_ID",
        how="left_semi"
    )
    .filter(F.col("SYMPTOM_ID") != symptome_id)
    .groupBy("SYMPTOM_ID")
    .agg(
        F.count("*").alias("nombre_associations")
    )
)

# ============================================================
# 7. Calculer la confiance et récupérer les noms
# ============================================================

resultat_symptome = (
    associations
    .join(
        symptoms,
        on="SYMPTOM_ID",
        how="left"
    )
    .select(
        F.col("SYMPTOM_NAME").alias("Symptôme"),
        (
            F.col("nombre_associations") /
            F.lit(support_symptome)
        ).alias("Information")
    )
    .orderBy("Symptôme")
).cache()

# ============================================================
# 8. Compter les lignes et mesurer le temps d'exécution
# ============================================================

nombre_lignes = resultat_symptome.count()

temps_execution = time.perf_counter() - debut

# ============================================================
# 9. Afficher les résultats
# ============================================================

print("RÉSULTATS POUR UN SYMPTÔME")
print(f"ID du symptôme : {symptome_id}")
print(f"Nom du symptôme : {nom_symptome}")
print(f"Nombre de lignes du tableau : {nombre_lignes}")
print(f"Temps d'exécution du calcul : {temps_execution:.2f} secondes")

resultat_symptome.show(20, truncate=False)

![Résultats Dizziness](./images/Capture%20d’écran%202026-10-29%20à%2019.23.38.png)

### Empreinte carbonne

Estimez l’empreinte carbone de votre exécution sur GCP à l’aide d’un calculateur d’empreinte carbone pour les workloads cloud. Indiquez les paramètres utilisés et le résultat obtenu (en kgCO₂e). Commentez brièvement votre résultat.

Vous pouvez utiliser le calculateur suivant: https://tools.finitizer.com/cloud-carbon-calculator, ou tout autre estimateur de votre choix.

L'empreinte carbone de l'exécution sur Google Cloud Platform (GCP) a été estimée à l'aide du Cloud Carbon Calculator de Finitizer.

### Paramètres utilisés

| **Paramètre** | **Valeur** |
|---|---|
| Fournisseur cloud | Google Cloud Platform (GCP) |
| Région | `us-east1` (Caroline du Sud, États-Unis) |
| Type de machine | `n4-highmem-16` |
| Nombre de machines | 3 (1 nœud pilote et 2 nœuds de calcul) |
| Processeurs par machine | 16 vCPU |
| Mémoire par machine | 128 GiB |
| Disque par machine | 150 Go |
| Début de la période | 28 octobre 2026 à 17 h 50 |
| Fin de la période | 29 octobre 2026 à 19 h 32 |
| Durée totale estimée | 25,7 heures |
| Transfert de données entre régions | 0 Go (hypothèse) |

### Résultat obtenu

Le calculateur affiche une empreinte carbone annualisée comprise entre 39 et 71 kg CO₂e, pour une utilisation de 25,7 heures par mois.

Pour obtenir l'empreinte correspondant à une seule période d'utilisation de 25,7 heures, le résultat annuel est divisé par 12 :

$$
E_{\min} = \frac{39}{12} = 3,25 \text{ kg CO}_2\text{e}
$$

$$
E_{\max} = \frac{71}{12} \approx 5,92 \text{ kg CO}_2\text{e}
$$

L'empreinte carbone estimée de l'exécution est donc comprise entre 3,25 et 5,92 kg CO₂e, soit environ 4,6 kg CO₂e en prenant le milieu de la fourchette.

### Commentaire

Cette estimation représente les émissions associées à l'utilisation des ressources de calcul du cluster Dataproc pendant la période considérée.

Le calcul repose sur l'hypothèse que trois machines virtuelles ont fonctionné pendant toute la période. Cependant, le cluster disposait d'une configuration d'autoscaling permettant d'ajouter des nœuds de calcul selon la charge de travail. Le nombre réel de machines utilisées a donc pu varier.

De plus, le stockage objet et les éventuels transferts de données entre régions non mesurés ne sont pas inclus dans cette estimation.

Le résultat doit ainsi être interprété comme une estimation approximative plutôt qu'une mesure exacte de l'empreinte carbone.